---

<div align="center">
  <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" width="80"/>
</div>

<h1 align="center">GenAI & Advanced Nets: Conceitos de RAG</h1>

<h3 align="center">PhD. Julles Mitoura</h3>

<div align="center">
  <img src="https://img.shields.io/badge/Python-3776AB?style=for-the-badge&logo=python&logoColor=white"/>
  <img src="https://img.shields.io/badge/Jupyter-F37626?style=for-the-badge&logo=jupyter&logoColor=white"/>
  <img src="https://img.shields.io/badge/OpenAI-412991?style=for-the-badge&logo=openai&logoColor=white"/>
</div>

---

In [1]:
# Obs: se você não estiver utilizando um ambiente virtual, instale as bibliotecas conforme se apresenta abaixo
%pip install -q -r requirements.txt

# pip é o gerenciador de pacotes do Python. Pense nele como o instalador oficial de libs Python.
# no notebook, usar %pip ... é ideal porque instala no mesmo ambiente do kernel em uso.

# -q: quiet
# -r: requirement file, indica ao pip para instalar os pacotes listados no arquivo requirements.txt

# Bibliotecas adicionadas neste notebook:
# pypdf  — extração de texto de arquivos PDF (pure Python, sem dependências nativas)
# numpy  — operações vetoriais (similaridade de cosseno, álgebra linear)
# openai — embeddings (text-embedding-3-small) e geração de texto (gpt-4o-mini)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.22.2 requires transformers>=4.55.0, but you have transformers 4.52.4 which is incompatible.

[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---

<div align="center">

## <span style="color:#1E90FF;">RAG: Visão Geral</span>

</div>

Na aula anterior aprendemos a consumir LLMs via API — enviando mensagens ao GPT e recebendo respostas. Mas esses modelos têm uma limitação fundamental: **eles só conhecem o que foi visto durante o treinamento**. Informações privadas, documentos internos ou publicações científicas recentes são invisíveis para eles.

**RAG (Retrieval-Augmented Generation)** resolve isso ao conectar o LLM a uma base de conhecimento externa, em tempo de inferência:

```
┌─────────────────────────────────────────────────────────────┐
│  Pergunta do usuário                                        │
│       ↓                                                     │
│  [1] RETRIEVE — busca os trechos mais relevantes do corpus  │
│       ↓                                                     │
│  [2] AUGMENT  — monta um prompt com a pergunta + contexto   │
│       ↓                                                     │
│  [3] GENERATE — LLM gera a resposta baseada no contexto     │
└─────────────────────────────────────────────────────────────┘
```

### Por que RAG?

| Problema | Solução com RAG |
|---|---|
| Knowledge cutoff — modelo não conhece dados recentes | Injeta documentos atualizados no prompt |
| Alucinações sobre domínio específico | Ancora a resposta em trechos reais do corpus |
| Fine-tuning é caro e estático | RAG atualiza a base sem re-treinar o modelo |
| Privacidade — não se pode enviar dados ao treino | Documentos ficam locais, só o contexto relevante vai à API |

Neste notebook construiremos **cada componente do zero**, usando apenas Python e NumPy. Ao final teremos um sistema RAG funcional que responde perguntas sobre um artigo científico real.

---

---

<div align="center">

## <span style="color:#1E90FF;">Setup: Configuração do Ambiente</span>

</div>

In [2]:
import os
import re
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
try:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print("Cliente OpenAI configurado com sucesso.")
except Exception as e:
    print(f"Erro ao configurar o cliente OpenAI: {e}")
    exit(1)

Cliente OpenAI configurado com sucesso.


---

<div align="center">

## <span style="color:#1E90FF;">O que são Embeddings?</span>

</div>

Um **embedding** é uma representação numérica de texto na forma de um vetor de números reais:

$$\vec{v} = \text{Embed}(\text{texto}) \in \mathbb{R}^n$$

onde $n$ é a dimensionalidade do espaço vetorial — no caso do `text-embedding-3-small`, $n = 1536$.

A propriedade fundamental é que **o significado semântico é preservado na geometria do espaço**: textos com significados parecidos produzem vetores próximos, enquanto textos de domínios distintos ficam distantes.

```
                       espaço vetorial R^1536

          "gato"  o-----------o "felino"      <- próximos (alta similaridade)



  "inteligência"  o                           <- distante (baixa similaridade)
  "artificial"
```

Internamente, o modelo de embedding foi treinado para que essa propriedade geométrica emerja a partir de enormes volumes de texto. O resultado é uma "tradução" de linguagem para geometria que podemos explorar computacionalmente.

### Modelos de Embedding OpenAI

| Modelo | Dimensões | Uso recomendado |
|---|---|---|
| `text-embedding-3-small` | 1536 | Alta eficiência, baixo custo — ideal para RAG |
| `text-embedding-3-large` | 3072 | Máxima qualidade semântica |
| `text-embedding-ada-002` | 1536 | Legado — substituído pelo small |

---

In [3]:
def embed(text: str) -> list[float]:
    """Gera o embedding de um texto usando o modelo text-embedding-3-small."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding


# Frases de exemplo para explorar propriedades dos embeddings
frases = [
    "Gatos são animais domésticos muito populares.",
    "Felinos são conhecidos por sua independência.",
    "A inteligência artificial está revolucionando a tecnologia.",
]

print("=" * 65)
for frase in frases:
    vetor = embed(frase)
    print(f"Texto     : {frase}")
    print(f"Dimensões : {len(vetor)}")
    print(f"Valores[0:8]: {[round(v, 5) for v in vetor[:8]]}")
    print("-" * 65)

Texto     : Gatos são animais domésticos muito populares.
Dimensões : 1536
Valores[0:8]: [0.05042, -0.00885, -0.0122, 0.04828, 0.02982, 0.03583, -0.05981, 0.02557]
-----------------------------------------------------------------
Texto     : Felinos são conhecidos por sua independência.
Dimensões : 1536
Valores[0:8]: [0.02972, 0.00214, 0.0052, 0.05557, 0.04437, 0.0361, -0.05435, 0.00213]
-----------------------------------------------------------------
Texto     : A inteligência artificial está revolucionando a tecnologia.
Dimensões : 1536
Valores[0:8]: [0.01817, 0.00686, -0.01924, 0.03549, 0.03949, -0.00243, -0.00307, 0.06134]
-----------------------------------------------------------------


---

<div align="center">

## <span style="color:#1E90FF;">Propriedades Semânticas dos Embeddings</span>

</div>

Os 1536 valores que acabamos de ver não fazem sentido individualmente — **o significado está na relação entre os vetores**, não nos valores isolados.

A propriedade central é:

> **Textos semanticamente similares → vetores geometricamente próximos**

Intuitivamente:
- *"Gatos são animais domésticos"* e *"Felinos são independentes"* falam sobre o mesmo ser — esperamos vetores próximos.
- *"Gatos"* e *"inteligência artificial"* são domínios completamente diferentes — esperamos vetores distantes.

Para medir essa proximidade precisamos de uma **métrica de similaridade entre vetores**, que definimos na próxima seção.

---

---

<div align="center">

## <span style="color:#1E90FF;">Similaridade de Vetores</span>

</div>

A métrica mais usada para comparar embeddings é a **similaridade de cosseno**. Em vez de medir a distância euclidiana (comprimento do segmento entre dois pontos), ela mede o **ângulo** entre os vetores — o que a torna invariante à magnitude e ideal para vetores de alta dimensão:

$$\text{cos\_sim}(\vec{u}, \vec{v}) = \frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\| \cdot \|\vec{v}\|} = \frac{\displaystyle\sum_{i=1}^{n} u_i v_i}{\sqrt{\displaystyle\sum_{i=1}^{n} u_i^2} \cdot \sqrt{\displaystyle\sum_{i=1}^{n} v_i^2}}$$

O numerador é o **produto escalar** (dot product) — mede o quanto os vetores "apontam na mesma direção". O denominador **normaliza** pelo produto das magnitudes, garantindo resultado no intervalo $[-1, 1]$.

### Interpretação

| Valor | Interpretação |
|---|---|
| `1.00` | Vetores idênticos — texto idêntico ou paráfrase perfeita |
| `0.70 – 0.95` | Alta similaridade semântica |
| `0.40 – 0.69` | Similaridade moderada — mesmo domínio amplo |
| `< 0.40` | Textos semanticamente distantes |
| `0.00` | Vetores ortogonais — sem relação semântica |

> Na prática, embeddings raramente atingem valores negativos pois os modelos são treinados para produzir vetores em configurações que evitam correlação negativa.

---

In [4]:
def cosine_similarity(u: np.ndarray, v: np.ndarray) -> float:
    """Calcula a similaridade de cosseno entre dois vetores NumPy."""
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))


# Gerar embeddings das frases de exemplo (como arrays NumPy)
vetores = {frase: np.array(embed(frase)) for frase in frases}

# Query para comparação
query = "Qual é o comportamento de animais de estimação?"
vetor_query = np.array(embed(query))

print(f'Query: "{query}"\n')
print(f'{"Frase":<52} {"Similaridade":>12}')
print("-" * 66)
for frase, vetor in vetores.items():
    sim = cosine_similarity(vetor_query, vetor)
    print(f"{frase:<52} {sim:>12.4f}")

print()
# Comparar as frases entre si
print("Similaridade entre frases do corpus:")
print("-" * 66)
frase_list = list(vetores.keys())
for i in range(len(frase_list)):
    for j in range(i + 1, len(frase_list)):
        sim = cosine_similarity(vetores[frase_list[i]], vetores[frase_list[j]])
        print(f"  [{i+1}] vs [{j+1}]: {sim:.4f}")
        print(f"    {frase_list[i]}")
        print(f"    {frase_list[j]}")
        print()

Query: "Qual é o comportamento de animais de estimação?"

Frase                                                Similaridade
------------------------------------------------------------------
Gatos são animais domésticos muito populares.              0.4826
Felinos são conhecidos por sua independência.              0.5037
A inteligência artificial está revolucionando a tecnologia.       0.2198

Similaridade entre frases do corpus:
------------------------------------------------------------------
  [1] vs [2]: 0.6210
    Gatos são animais domésticos muito populares.
    Felinos são conhecidos por sua independência.

  [1] vs [3]: 0.2919
    Gatos são animais domésticos muito populares.
    A inteligência artificial está revolucionando a tecnologia.

  [2] vs [3]: 0.2548
    Felinos são conhecidos por sua independência.
    A inteligência artificial está revolucionando a tecnologia.



---

<div align="center">

## <span style="color:#1E90FF;">Banco de Dados Vetorial Local</span>

</div>

Em produção, bancos de dados vetoriais como **FAISS**, **ChromaDB** e **Pinecone** indexam milhões de embeddings com estruturas de dados especializadas (HNSW, IVF) para busca aproximada em milissegundos.

Neste notebook implementamos um `VectorDatabase` **minimalista em Python puro** — sem dependências externas — que usa **busca linear** com similaridade de cosseno. Isso é pedagogicamente valioso: desmistifica o que um banco vetorial faz e é suficiente para corpora de até alguns milhares de documentos.

### Interface da Classe

| Método | Assinatura | Descrição |
|---|---|---|
| `add` | `(id, text, embedding)` | Armazena um documento com seu embedding |
| `search` | `(query_embedding, top_k)` | Retorna os `top_k` documentos mais similares |
| `__len__` | — | Número de documentos indexados |

A busca linear tem complexidade **O(n)** — percorre todos os documentos a cada consulta. Na Aula 8 substituiremos isso por approximate nearest neighbors (ANN).

---

In [5]:
class VectorDatabase:
    """
    Banco de dados vetorial minimalista usando listas Python + NumPy.
    Busca linear com similaridade de cosseno — O(n) por consulta.
    Adequado para corpora de até alguns milhares de documentos.
    """

    def __init__(self):
        self._ids: list[str] = []
        self._texts: list[str] = []
        self._embeddings: list[np.ndarray] = []

    def add(self, doc_id: str, text: str, embedding: list[float]) -> None:
        """Adiciona um documento ao banco de dados."""
        self._ids.append(doc_id)
        self._texts.append(text)
        self._embeddings.append(np.array(embedding))

    def search(self, query_embedding: np.ndarray, top_k: int = 3) -> list[dict]:
        """
        Retorna os top_k documentos mais similares à query.

        Returns:
            Lista de dicts com chaves: 'id', 'text', 'score'.
        """
        scores = [
            cosine_similarity(query_embedding, emb)
            for emb in self._embeddings
        ]
        ranked = sorted(
            zip(self._ids, self._texts, scores),
            key=lambda x: x[2],
            reverse=True
        )
        return [
            {"id": id_, "text": text, "score": score}
            for id_, text, score in ranked[:top_k]
        ]

    def __len__(self) -> int:
        return len(self._ids)


# --- Demo com frases de exemplo ---
db_demo = VectorDatabase()

docs_demo = [
    ("d1", "Gatos são animais domésticos muito populares."),
    ("d2", "Felinos são conhecidos por sua independência."),
    ("d3", "A inteligência artificial está revolucionando a tecnologia."),
    ("d4", "Python é uma linguagem de programação versátil."),
    ("d5", "Redes neurais são a base do deep learning."),
]

print("Indexando documentos de exemplo...")
for doc_id, text in docs_demo:
    db_demo.add(doc_id, text, embed(text))
print(f"Banco de dados com {len(db_demo)} documentos.\n")

# Consulta
query_demo = "Quais animais são mantidos como pets?"
resultados = db_demo.search(np.array(embed(query_demo)), top_k=3)

print(f'Query: "{query_demo}"\n')
print(f'{"ID":<6} {"Score":>8}  Texto')
print("-" * 70)
for r in resultados:
    print(f"  {r['id']}   {r['score']:.4f}  {r['text']}")

Indexando documentos de exemplo...
Banco de dados com 5 documentos.

Query: "Quais animais são mantidos como pets?"

ID        Score  Texto
----------------------------------------------------------------------
  d1   0.5479  Gatos são animais domésticos muito populares.
  d2   0.4223  Felinos são conhecidos por sua independência.
  d4   0.1695  Python é uma linguagem de programação versátil.


---

<div align="center">

## <span style="color:#1E90FF;">Extração de Texto do PDF</span>

</div>

Para construir o RAG precisamos de um corpus. Usaremos um **artigo científico real** como fonte de conhecimento:

> *"A Python-Based Thermodynamic Equilibrium Library for Gibbs Energy Minimization: A Case Study on Supercritical Water Gasification of Ethanol and Methanol"*  
> — *Eng* 2025, 6, 208. doi: 10.3390/eng6090208

Para extrair o texto usamos a biblioteca `pypdf`, que lê PDFs em Python puro sem depender de ferramentas externas como Poppler ou Ghostscript.

```python
from pypdf import PdfReader
reader = PdfReader("arquivo.pdf")
texto = reader.pages[0].extract_text()
```

### Limpeza do Texto

PDFs científicos têm artefatos típicos que precisamos remover:

| Artefato | Exemplo | Regex |
|---|---|---|
| Cabeçalho do periódico | `Eng 2025, 6, 208` | `Eng \\d+, \\d+, \\d+` |
| Número de página | `3 of 21` | `\\d+ of \\d+` |
| Hifenização de linha | `compu-\\ntacional` | `-\\n(\\w)` |
| Espaços múltiplos | `texto   com` | ` {2,}` |

---

In [6]:
from pypdf import PdfReader

PDF_PATH = "docs/article.pdf"

# --- Extração ---
reader = PdfReader(PDF_PATH)
print(f"Número de páginas: {len(reader.pages)}")

raw_pages = [page.extract_text() or "" for page in reader.pages]
raw_text = "\n".join(raw_pages)
print(f"Total de caracteres (bruto): {len(raw_text)}")

print("\n=== Primeiros 500 caracteres (bruto) ===")
print(raw_text[:500])


# --- Limpeza ---
def clean_text(text: str) -> str:
    """
    Remove artefatos comuns de extração de PDF:
    - Cabeçalhos de página do periódico (ex: 'Eng 2025, 6, 208')
    - Indicadores de página (ex: '3 of 21')
    - Hifenização de final de linha (re-junta palavras quebradas)
    - Espaços e quebras de linha excessivos
    """
    text = re.sub(r"Eng \d+, \d+, \d+\s*\n?\d* of \d+\n?", "", text)
    text = re.sub(r"-\n(\w)", r"\1", text)          # re-junta hifenização
    text = re.sub(r" {2,}", " ", text)               # espaços múltiplos
    text = re.sub(r"\n{3,}", "\n\n", text)           # quebras excessivas
    return text.strip()


clean = clean_text(raw_text)
print(f"\nTotal de caracteres (limpo): {len(clean)}")
print("\n=== Primeiros 500 caracteres (limpo) ===")
print(clean[:500])

Número de páginas: 21
Total de caracteres (bruto): 61502

=== Primeiros 500 caracteres (bruto) ===
Academic Editor: Stanisław
Wacławek
Received: 24 July 2025
Revised: 11 August 2025
Accepted: 28 August 2025
Published: 30 August 2025
Citation: dos Santos Junior, J.M.;
Daltro de Freitas, A.C.; Mariano, A.P .
A Python-Based Thermodynamic
Equilibrium Library for Gibbs Energy
Minimization: A Case Study on
Supercritical Water Gasification of
Ethanol and Methanol. Eng 2025, 6,
208. https://doi.org/10.3390/
eng6090208
Copyright: © 2025 by the authors.
Licensee MDPI, Basel, Switzerland.
This article i

Total de caracteres (limpo): 60799

=== Primeiros 500 caracteres (limpo) ===
Academic Editor: Stanisław
Wacławek
Received: 24 July 2025
Revised: 11 August 2025
Accepted: 28 August 2025
Published: 30 August 2025
Citation: dos Santos Junior, J.M.;
Daltro de Freitas, A.C.; Mariano, A.P .
A Python-Based Thermodynamic
Equilibrium Library for Gibbs Energy
Minimization: A Case Study on
Supercritical Wat

---

<div align="center">

## <span style="color:#1E90FF;">Chunking: Segmentação de Texto</span>

</div>

Não podemos passar o artigo inteiro (21 páginas, ~50.000 caracteres) como contexto para o LLM por dois motivos:

1. **Janela de contexto limitada** — cada chamada à API tem um limite de tokens.
2. **Granularidade de recuperação** — embeddings de documentos inteiros capturam temas gerais, não detalhes específicos. Chunks menores permitem recuperar **o trecho exato** que responde a uma pergunta.

### Estratégia: Divisão por Caracteres com Sobreposição

Dividimos o texto em janelas deslizantes de tamanho fixo, com **overlap** (sobreposição) entre chunks consecutivos:

```
|<------ chunk 0 (1000 chars) ------>|
                    |<- overlap ->|
                    |<------ chunk 1 (1000 chars) ------>|
                                        |<- overlap ->|
                                        |<-- chunk 2 -->|
```

O overlap garante que **frases importantes na fronteira entre chunks não sejam cortadas** — o início do chunk 1 contém o final do chunk 0, preservando contexto.

### Parâmetros

| Parâmetro | Valor | Critério |
|---|---|---|
| `chunk_size` | 1000 chars | ~200 tokens — suficiente para um parágrafo completo |
| `overlap` | 200 chars | 20% do chunk — preserva contexto de fronteira |

---

In [7]:
def chunk_text(text: str, chunk_size: int = 1000, overlap: int = 200) -> list[str]:
    """
    Divide texto em chunks de tamanho fixo com sobreposição.

    Args:
        text: Texto limpo a ser segmentado.
        chunk_size: Número de caracteres por chunk.
        overlap: Número de caracteres sobrepostos entre chunks consecutivos.

    Returns:
        Lista de strings representando os chunks.
    """
    if chunk_size <= overlap:
        raise ValueError("chunk_size deve ser maior que overlap.")

    chunks = []
    step = chunk_size - overlap
    start = 0
    while start < len(text):
        chunks.append(text[start : start + chunk_size])
        start += step
    return chunks


chunks = chunk_text(clean, chunk_size=1000, overlap=200)

print(f"Total de chunks   : {len(chunks)}")
print(f"chunk_size        : 1000 chars")
print(f"overlap           : 200 chars")
print(f"Tamanho médio     : {sum(len(c) for c in chunks) / len(chunks):.0f} chars")

print("\n=== Chunk 0 ===")
print(chunks[0])

print("\n=== Chunk 1 ===")
print(chunks[1])

print("\n=== Verificação do overlap ===")
overlap_ok = chunks[0][-200:] == chunks[1][:200]
print(f"Últimos 200 chars do chunk 0 == primeiros 200 chars do chunk 1: {overlap_ok}")

Total de chunks   : 76
chunk_size        : 1000 chars
overlap           : 200 chars
Tamanho médio     : 997 chars

=== Chunk 0 ===
Academic Editor: Stanisław
Wacławek
Received: 24 July 2025
Revised: 11 August 2025
Accepted: 28 August 2025
Published: 30 August 2025
Citation: dos Santos Junior, J.M.;
Daltro de Freitas, A.C.; Mariano, A.P .
A Python-Based Thermodynamic
Equilibrium Library for Gibbs Energy
Minimization: A Case Study on
Supercritical Water Gasification of
Ethanol and Methanol. Eng 2025, 6,
208. https://doi.org/10.3390/
eng6090208
Copyright: © 2025 by the authors.
Licensee MDPI, Basel, Switzerland.
This article is an open access article
distributed under the terms and
conditions of the Creative Commons
Attribution (CC BY) license
(https://creativecommons.org/
licenses/by/4.0/).
Article
A Python-Based Thermodynamic Equilibrium Library for Gibbs
Energy Minimization: A Case Study on Supercritical Water
Gasification of Ethanol and Methanol
Julles Mitoura dos Santos Junior 1, Ant

---

<div align="center">

## <span style="color:#1E90FF;">Pipeline RAG Completo</span>

</div>

Reunindo todos os componentes construídos nas seções anteriores. O pipeline RAG opera em **duas fases distintas**:

### Fase 1 — Indexação (executada uma vez)

```
docs/article.pdf
      |  pypdf.PdfReader
      v
  texto bruto
      |  clean_text()
      v
  texto limpo
      |  chunk_text(size=1000, overlap=200)
      v
  [chunk_0, chunk_1, ..., chunk_N]
      |  embed(chunk_i) — uma chamada à API por chunk
      v
  VectorDatabase indexado
```

### Fase 2 — Consulta (executada por pergunta)

```
  pergunta do usuário
      |  embed(pergunta)
      v
  vetor da query
      |  VectorDatabase.search(top_k=3)
      v
  [chunk_mais_relevante, chunk_2, chunk_3]
      |  concatenar como contexto
      v
  prompt = system_prompt + contexto + pergunta
      |  gpt-4o-mini
      v
  resposta fundamentada no artigo
```

---

In [8]:
# ============================================================
# FASE 1: INDEXAÇÃO
# Embeda todos os chunks e armazena no VectorDatabase.
# Esta etapa faz ~N chamadas à API de embeddings (uma por chunk).
# Em produção, o índice seria salvo em disco e reutilizado.
# ============================================================

print("Iniciando indexação do artigo científico...")
print(f"Total de chunks a indexar: {len(chunks)}\n")

db = VectorDatabase()

for i, chunk in enumerate(chunks):
    chunk_id = f"chunk_{i:03d}"
    embedding = embed(chunk)
    db.add(chunk_id, chunk, embedding)
    if (i + 1) % 10 == 0 or i == len(chunks) - 1:
        print(f"  Indexados: {i + 1}/{len(chunks)} chunks")

print(f"\nIndexação concluída. Banco de dados com {len(db)} documentos.")

Iniciando indexação do artigo científico...
Total de chunks a indexar: 76

  Indexados: 10/76 chunks
  Indexados: 20/76 chunks
  Indexados: 30/76 chunks
  Indexados: 40/76 chunks
  Indexados: 50/76 chunks
  Indexados: 60/76 chunks
  Indexados: 70/76 chunks
  Indexados: 76/76 chunks

Indexação concluída. Banco de dados com 76 documentos.


In [9]:
# ============================================================
# FASE 2: FUNÇÃO DE CONSULTA RAG
# ============================================================

def rag_query(question: str, top_k: int = 3, verbose: bool = False) -> str:
    """
    Executa uma consulta RAG completa:
      1. Gera embedding da pergunta
      2. Recupera os top_k chunks mais relevantes do VectorDatabase
      3. Constrói um prompt aumentado com o contexto recuperado
      4. Chama gpt-4o-mini para gerar a resposta fundamentada

    Args:
        question: Pergunta em linguagem natural.
        top_k: Número de chunks a recuperar (mais chunks = mais contexto, mais tokens).
        verbose: Se True, exibe os chunks recuperados antes da resposta.

    Returns:
        Resposta gerada pelo LLM, fundamentada nos chunks recuperados.
    """
    # Passo 1: embedding da pergunta
    query_embedding = np.array(embed(question))

    # Passo 2: recuperação dos chunks mais relevantes
    resultados = db.search(query_embedding, top_k=top_k)

    if verbose:
        print(f"=== Chunks recuperados (top {top_k}) ===")
        for r in resultados:
            print(f"  [{r['id']}]  score={r['score']:.4f}")
            print(f"  {r['text'][:120].strip()}...")
            print()

    # Passo 3: construção do contexto aumentado
    # Separador explícito entre chunks para o LLM identificar fronteiras
    contexto = "\n\n---\n\n".join(r["text"] for r in resultados)

    # Passo 4: chamada ao LLM com prompt aumentado
    system_prompt = (
        "Você é um assistente especializado em termodinâmica computacional e Python científico. "
        "Responda à pergunta do usuário com base EXCLUSIVAMENTE no contexto fornecido. "
        "Se a informação não estiver no contexto, informe que não encontrou no artigo. "
        "Seja preciso, técnico e cite dados numéricos quando disponíveis no contexto."
    )

    user_message = (
        f"Contexto extraído do artigo científico:\n\n{contexto}\n\n"
        f"Pergunta: {question}"
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.1,  # baixa temperatura para respostas factuais
    )

    return response.choices[0].message.content

---

<div align="center">

## <span style="color:#1E90FF;">Demonstração: Perguntas sobre o Artigo</span>

</div>

Testamos o sistema RAG com três perguntas sobre o artigo, cobrindo diferentes seções e tipos de informação:

| # | Pergunta | Seção do artigo | Tipo |
|---|---|---|---|
| 1 | Equações de estado suportadas | Metodologia (p. 5) | Factual — metodologia |
| 2 | Eficiência de gaseificação EtOH vs MeOH | Introdução (p. 3) | Factual — resultado numérico |
| 3 | Efeito da temperatura na produção H₂/CH₄ | Resultados (p. 8–12) | Analítico — tendência |

---

In [11]:
perguntas = [
#    "Quais equações de estado são suportadas pela biblioteca tes-thermo para modelar as não-idealidades do sistema reativo?",
#    "Qual é a eficiência de gaseificação do etanol comparada à do metanol a 550°C e 25 MPa, de acordo com o artigo?",
    "Como a temperatura influencia a produção de hidrogênio e metano no processo de gaseificação supercrítica do etanol (SCWG)?",
]

for i, pergunta in enumerate(perguntas, start=1):
    print("=" * 72)
    print(f"PERGUNTA {i}: {pergunta}")
    print("=" * 72)
    resposta = rag_query(pergunta, top_k=3, verbose=True)
    print(f"RESPOSTA:\n{resposta}")
    print()

PERGUNTA 1: Como a temperatura influencia a produção de hidrogênio e metano no processo de gaseificação supercrítica do etanol (SCWG)?
=== Chunks recuperados (top 3) ===
  [chunk_037]  score=0.6763
  ied conditions. This result can be attributed to the excess water in the feed, which
promotes higher H2 production by sh...

  [chunk_043]  score=0.6726
  anol
SCWG process.
On the other hand, pressure has a negligible influence on H2 yield under the studied
conditions, whic...

  [chunk_038]  score=0.6524
  ts of reactor system temperature and ethanol feed concentration (wt%) on
product formation: (a) methane, (b) carbon diox...

RESPOSTA:
A temperatura tem um impacto significativo na produção de hidrogênio e metano no processo de gaseificação supercrítica do etanol (SCWG). 

Para a produção de hidrogênio (H2), a análise indica que a temperatura influencia a relação etanol/água, que mostra uma correlação positiva fraca com a produção de H2. Em temperaturas mais baixas, essa relação tem po

---

<div align="center">

## <span style="color:#1E90FF;">Resumo e Próximos Passos</span>

</div>

Neste notebook construímos um pipeline RAG completo do zero, sem nenhum framework externo:

| Componente | Implementação | Propósito |
|---|---|---|
| Embeddings | `text-embedding-3-small` (OpenAI API) | Representação semântica de texto |
| Similaridade | Cosseno em NumPy puro | Medida de proximidade vetorial |
| Vector DB | `VectorDatabase` (listas + NumPy) | Armazenamento e busca linear |
| Extração | `pypdf.PdfReader` | Leitura do corpus PDF |
| Chunking | Divisão por caracteres com overlap | Granularidade para recuperação |
| Geração | `gpt-4o-mini` (OpenAI API) | Síntese da resposta com contexto |

### Limitações da Implementação Atual

| Limitação | Impacto | Solução |
|---|---|---|
| Busca linear **O(n)** | Inviável para > 10.000 docs | FAISS / ChromaDB com ANN |
| Chunking fixo por caracteres | Pode cortar frases no meio | Chunking por parágrafo / semântico |
| Sem reranking | Top-k não é sempre o mais relevante | Cross-encoder para refinamento |
| Índice em memória | Perdido ao reiniciar o kernel | Persistência em disco / banco |

### Na Próxima Aula (Aula 8)

Substituiremos o `VectorDatabase` artesanal por bancos de dados vetoriais de produção:

- **FAISS** — biblioteca do Meta para busca por approximate nearest neighbors (ANN) em escala de bilhões de vetores.
- **ChromaDB** — banco vetorial open-source com persistência em disco, metadados e filtros.

---